
# Yambda 12 · Export SASRec candidates for the ranker (Colab GPU)

Stage 1 of the two-stage system needs each source's top-200 unheard tracks per user.
iALS and popularity are computed locally; SASRec needs the GPU, so its lists are exported here:

* **validation**: from the model trained in notebook 08 (`sasrec_val.pt`), no retraining
* **test**: SASRec retrained on the test split with the validation settings (12 epochs), as in notebook 10

Needs in `MyDrive/music_recsys/`: `colab_bundle.zip`, the **new** `colab_src.zip`, and `outputs/sasrec_val.pt`.
Runtime → T4 GPU. ~12 min.


In [ ]:

from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/music_recsys"
!rm -rf /content/music_recsys && mkdir -p /content/music_recsys
!unzip -q "{DRIVE}/colab_bundle.zip" -d /content/music_recsys
!unzip -q -o "{DRIVE}/colab_src.zip" -d /content/music_recsys
!mkdir -p /content/music_recsys/reports/yambda && cp "{DRIVE}/outputs/sasrec_val.pt" "{DRIVE}/outputs/best_sasrec.json" /content/music_recsys/reports/yambda/

import sys
sys.path.insert(0, "/content/music_recsys/src")
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE (switch runtime!)")


In [ ]:

import json, time
import pandas as pd
from musicrec import config
from musicrec.preprocess import Split
from musicrec.evaluation import evaluate
from musicrec.models.sasrec import SASRec
from musicrec.ranking import candidates_from_model

OUT = config.PROJECT_ROOT / "reports" / "yambda"
K_CANDS = 200


## Validation candidates (trained model from notebook 08)

In [ ]:

val = Split.load(config.YAMBDA_SPLIT_VAL_DIR)
sas_val = SASRec.from_checkpoint(OUT / "sasrec_val.pt", val)
print("sanity, should be ~0.0268:", round(evaluate(sas_val, val)["recall@10"], 4))
cands_val = candidates_from_model(sas_val, val, K_CANDS)
cands_val.to_parquet(OUT / "cands_val_sasrec.parquet", index=False)
cands_val.shape


## Test candidates (retrain on the test split with validation settings)

In [ ]:

test = Split.load(config.YAMBDA_SPLIT_TEST_DIR)
v1 = json.loads((OUT / "best_sasrec.json").read_text())
sas_test = SASRec(**v1["params"], epochs=v1["best_epoch"], early_stopping=False)
t = time.time(); sas_test.fit(test); print(f"trained in {time.time() - t:.0f}s")
print("test recall@10 (notebook 10 reported 0.0227; small GPU nondeterminism is expected):",
      round(evaluate(sas_test, test)["recall@10"], 4))
cands_test = candidates_from_model(sas_test, test, K_CANDS)
cands_test.to_parquet(OUT / "cands_test_sasrec.parquet", index=False)
torch.save({"state_dict": sas_test.module_.state_dict(), "params": sas_test.params(), "best_epoch": v1["best_epoch"]},
           OUT / "sasrec_test.pt")


In [ ]:

!mkdir -p "{DRIVE}/outputs_cands" && cp {OUT}/cands_val_sasrec.parquet {OUT}/cands_test_sasrec.parquet {OUT}/sasrec_test.pt "{DRIVE}/outputs_cands/"
!ls -la "{DRIVE}/outputs_cands"
